# 100k Company Accounts Matching and Financial Coverage

This notebook matches the selected 100k Companies House company list to monthly accounts bulk files covering 2024-07 to 2026-06.

Main outputs:

- how many selected companies have matched accounts filings
- how many companies have observed turnover
- filing-level and company-level evidence tier distribution
- how many companies have at least two disclosed time points
- coverage by account category and sector

The notebook reads ZIP central directories first and only parses accounts files matched to the 100k company list. It does not extract all ZIP contents to disk.

## 1. Configuration

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import json
import math
import re
import time
import zipfile
import numpy as np
import pandas as pd

COMPANY_CSV = Path(r"E:\000硕士毕设\Haoran_CHENG\01 Data PreProcessing\01_CompaniesSelected\UKcompanies_active_account_category_sample_100k.csv")
ACCOUNTS_ZIP_DIR = Path(r"E:\000硕士毕设\财务数据\Local Large Data\Accounts Data_2024.7_2026.6")
OUTPUT_DIR = Path(r"E:\000硕士毕设\公司+财务数据匹配")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CSV_ENCODING = "utf-8-sig"
WRITE_FACTS_LONG = False  # Set True if you want a debug-level fact table. It can be large.
MAX_PARSE_FILES = None    # Set an integer for a quick smoke test, e.g. 500. Use None for full run.

MATCH_INDEX_CSV = OUTPUT_DIR / "accounts_match_index_100k.csv"
FILING_FEATURES_CSV = OUTPUT_DIR / "financial_features_filing_100k.csv"
COMPANY_SUMMARY_CSV = OUTPUT_DIR / "financial_features_company_summary_100k.csv"
TURNOVER_COMPANIES_CSV = OUTPUT_DIR / "turnover_companies_100k.csv"
MULTI_PERIOD_COMPANIES_CSV = OUTPUT_DIR / "multi_period_companies_100k.csv"
TIER_DISTRIBUTION_CSV = OUTPUT_DIR / "financial_evidence_tier_distribution_100k.csv"
CATEGORY_COVERAGE_CSV = OUTPUT_DIR / "account_category_financial_coverage_100k.csv"
SECTOR_COVERAGE_CSV = OUTPUT_DIR / "sector_financial_coverage_100k.csv"
SUMMARY_JSON = OUTPUT_DIR / "accounts_financial_matching_summary_100k.json"
FACTS_LONG_CSV = OUTPUT_DIR / "financial_facts_long_100k.csv"

print("Company CSV:", COMPANY_CSV)
print("Accounts ZIP dir:", ACCOUNTS_ZIP_DIR)
print("Output dir:", OUTPUT_DIR)

Company CSV: E:\000硕士毕设\Haoran_CHENG\01 Data PreProcessing\01_CompaniesSelected\UKcompanies_active_account_category_sample_100k.csv
Accounts ZIP dir: E:\000硕士毕设\财务数据\Local Large Data\Accounts Data_2024.7_2026.6
Output dir: E:\000硕士毕设\公司+财务数据匹配


## 2. Helper Functions

In [2]:
def normalise_company_number(value):
    if pd.isna(value):
        return ""
    s = str(value).strip().upper()
    s = re.sub(r"[^A-Z0-9]", "", s)
    if not s:
        return ""
    if s.isdigit():
        return s.zfill(8)
    # Letter-prefixed UK company numbers are normally 8 characters, e.g. SC123456.
    if len(s) < 8 and re.match(r"^[A-Z]{2}\d+$", s):
        return s[:2] + s[2:].zfill(6)
    return s


def parse_member_name(member_name):
    name = Path(member_name).name
    m = re.search(r"[_-]([A-Z]{0,2}\d{6,8})[_-](\d{8})\.(html|xml|zip)$", name, flags=re.IGNORECASE)
    if not m:
        return None
    company_number = normalise_company_number(m.group(1))
    raw_date = m.group(2)
    period_end = f"{raw_date[:4]}-{raw_date[4:6]}-{raw_date[6:8]}"
    return company_number, period_end, m.group(3).lower()


def attrs_to_dict(attr_text):
    attrs = {}
    for key, value in re.findall(r"([\w:.-]+)\s*=\s*['\"]([^'\"]*)['\"]", attr_text or ""):
        attrs[key.lower()] = value
    return attrs


def strip_tags(text):
    text = re.sub(r"<[^>]+>", "", text or "")
    text = text.replace("&nbsp;", " ").replace("&#160;", " ")
    text = text.replace("&amp;", "&")
    return text.strip()


def parse_numeric(raw_value, scale=None, sign=None):
    if raw_value is None:
        return np.nan
    s = strip_tags(str(raw_value))
    s = s.replace(",", "").replace("£", "").replace("$", "")
    s = re.sub(r"\s+", "", s)
    if s in {"", "-", "—", "nan", "None"}:
        return np.nan
    negative_by_parentheses = s.startswith("(") and s.endswith(")")
    s = s.strip("()")
    try:
        value = float(s)
    except Exception:
        return np.nan
    if negative_by_parentheses:
        value = -value
    if sign == "-":
        value = -value
    try:
        if scale not in {None, ""}:
            value = value * (10 ** int(scale))
    except Exception:
        pass
    return value


def extract_context_info(text):
    context_info = {}
    for m in re.finditer(r"<(?:\w+:)?context\b([^>]*)>(.*?)</(?:\w+:)?context>", text, flags=re.IGNORECASE | re.DOTALL):
        attrs = attrs_to_dict(m.group(1))
        context_id = attrs.get("id")
        body = m.group(2)
        if not context_id:
            continue
        end_match = re.search(r"<(?:\w+:)?endDate>(.*?)</(?:\w+:)?endDate>", body, flags=re.IGNORECASE | re.DOTALL)
        instant_match = re.search(r"<(?:\w+:)?instant>(.*?)</(?:\w+:)?instant>", body, flags=re.IGNORECASE | re.DOTALL)
        end_date = strip_tags(end_match.group(1)) if end_match else (strip_tags(instant_match.group(1)) if instant_match else "")
        dimension_count = len(re.findall(r"(?:explicitMember|typedMember)", body, flags=re.IGNORECASE))
        context_info[context_id] = {"context_end_date": end_date, "context_dimension_count": dimension_count}
    return context_info


METRIC_TAG_PATTERNS = {
    "turnover": [
        r"TurnoverRevenue$", r"^Turnover$", r"RevenueFromSaleOfGoods$", r"RevenueFromRenderingServices$",
        r"RevenueFromContractsWithCustomers$", r"GrossOperatingRevenue$",
    ],
    "current_assets": [r"CurrentAssets$"],
    "fixed_assets": [r"FixedAssets$"],
    "net_current_assets_liabilities": [r"NetCurrentAssetsLiabilities$"],
    "total_assets_less_current_liabilities": [r"TotalAssetsLessCurrentLiabilities$"],
    "net_assets_liabilities": [r"NetAssetsLiabilities$", r"NetAssets$"],
    "equity": [r"Equity$", r"ShareholderFunds$", r"CapitalAndReserves$"],
    "cash": [r"CashBankOnHand$", r"CashAndCashEquivalents$", r"CashAtBankAndInHand$"],
    "debtors": [r"Debtors$", r"DebtorsAmountsFallingDueWithinOneYear$"],
    "creditors_within_one_year": [r"CreditorsAmountsFallingDueWithinOneYear$", r"CreditorsDueWithinOneYear$"],
    "creditors_after_one_year": [r"CreditorsAmountsFallingDueAfterMoreThanOneYear$", r"CreditorsDueAfterOneYear$", r"CreditorsAmountsFallingDueAfterOneYear$"],
    "creditors_total": [r"Creditors$", r"TotalCreditors$"],
    "employees": [r"AverageNumberEmployeesDuringPeriod$", r"AverageNumberOfEmployeesDuringPeriod$", r"EmployeesTotal$"],
    "profit_loss": [r"ProfitLoss$", r"ProfitLossBeforeTax$", r"OperatingProfitLoss$"],
}

EXCLUDE_FACT_NAME_PATTERNS = [
    r"Policy", r"Description", r"Disclosure", r"Narrative", r"TextBlock", r"Taxonomy",
    r"RevenueRecognition", r"DeferredTax", r"IncomeTax", r"TaxCredit", r"CostOfSales",
]


def metric_for_fact_name(fact_name):
    local = fact_name.split(":")[-1]
    if any(re.search(p, local, flags=re.IGNORECASE) for p in EXCLUDE_FACT_NAME_PATTERNS):
        return None
    for metric, patterns in METRIC_TAG_PATTERNS.items():
        for pattern in patterns:
            if re.search(pattern, local, flags=re.IGNORECASE):
                return metric
    return None


def selection_score(context_end_date, period_end, dimension_count):
    score = 0
    if context_end_date and period_end and str(context_end_date)[:10] != str(period_end)[:10]:
        score += 20
    score += int(dimension_count or 0) * 5
    return score


def parse_financial_facts(text, company_number, period_end, source_zip, internal_filename):
    context_info = extract_context_info(text)
    facts = []

    # iXBRL nonFraction facts.
    ix_pattern = re.compile(r"<ix:(?:nonFraction|nonNumeric)\b([^>]*)>(.*?)</ix:(?:nonFraction|nonNumeric)>", re.IGNORECASE | re.DOTALL)
    for m in ix_pattern.finditer(text):
        attrs = attrs_to_dict(m.group(1))
        fact_name = attrs.get("name", "")
        metric = metric_for_fact_name(fact_name)
        if not metric:
            continue
        context_ref = attrs.get("contextref", attrs.get("contextRef", ""))
        ctx = context_info.get(context_ref, {})
        value = parse_numeric(m.group(2), scale=attrs.get("scale"), sign=attrs.get("sign"))
        if pd.isna(value):
            continue
        facts.append({
            "CompanyNumber": company_number,
            "period_end": period_end,
            "source_zip": source_zip,
            "internal_filename": internal_filename,
            "metric": metric,
            "fact_name": fact_name,
            "numeric_value": value,
            "context_ref": context_ref,
            "context_end_date": ctx.get("context_end_date", ""),
            "context_dimension_count": ctx.get("context_dimension_count", 0),
            "selection_score": selection_score(ctx.get("context_end_date", ""), period_end, ctx.get("context_dimension_count", 0)),
        })

    # Generic XML/XBRL facts, useful when files are not iXBRL HTML.
    xml_pattern = re.compile(r"<([A-Za-z0-9_:\.-]+)\b([^>]*)>([^<>{}]{1,200})</\1>", re.IGNORECASE | re.DOTALL)
    for m in xml_pattern.finditer(text):
        fact_name = m.group(1)
        metric = metric_for_fact_name(fact_name)
        if not metric:
            continue
        attrs = attrs_to_dict(m.group(2))
        context_ref = attrs.get("contextref", attrs.get("contextRef", ""))
        ctx = context_info.get(context_ref, {})
        value = parse_numeric(m.group(3), scale=attrs.get("scale"), sign=attrs.get("sign"))
        if pd.isna(value):
            continue
        facts.append({
            "CompanyNumber": company_number,
            "period_end": period_end,
            "source_zip": source_zip,
            "internal_filename": internal_filename,
            "metric": metric,
            "fact_name": fact_name,
            "numeric_value": value,
            "context_ref": context_ref,
            "context_end_date": ctx.get("context_end_date", ""),
            "context_dimension_count": ctx.get("context_dimension_count", 0),
            "selection_score": selection_score(ctx.get("context_end_date", ""), period_end, ctx.get("context_dimension_count", 0)),
        })

    return facts


def select_metric_values(facts):
    if not facts:
        return {}
    df = pd.DataFrame(facts)
    selected = {}
    for metric, g in df.sort_values(["metric", "selection_score"]).groupby("metric", sort=False):
        selected[metric] = float(g.iloc[0]["numeric_value"])
    if "creditors_total" not in selected:
        within = selected.get("creditors_within_one_year")
        after = selected.get("creditors_after_one_year")
        if within is not None or after is not None:
            selected["creditors_total"] = float((within or 0) + (after or 0))
    return selected


CORE_PROXY_FIELDS = ["current_assets", "net_assets_liabilities", "equity", "creditors_total", "employees", "cash", "debtors"]


def assign_evidence_tier(row):
    if pd.notna(row.get("turnover")):
        return "T1_observed_turnover"
    core_count = int(row.get("available_core_proxy_field_count", 0) or 0)
    if core_count >= 4:
        return "T2_balance_sheet_rich"
    if core_count >= 2:
        return "T3_balance_sheet_partial"
    return "T4_account_category_only"


def safe_read_text_from_zip(zf, entry):
    raw = zf.read(entry)
    for enc in ("utf-8", "utf-8-sig", "latin-1"):
        try:
            return raw.decode(enc)
        except Exception:
            continue
    return raw.decode("latin-1", errors="replace")

## 3. Read Selected 100k Companies

In [3]:
companies = pd.read_csv(COMPANY_CSV, dtype=str, encoding=CSV_ENCODING, low_memory=False)
companies["CompanyNumber_norm"] = companies["CompanyNumber"].map(normalise_company_number)
companies = companies.drop_duplicates("CompanyNumber_norm").copy()
target_company_numbers = set(companies["CompanyNumber_norm"].dropna())

print("Selected companies:", len(companies))
print("Unique target company numbers:", len(target_company_numbers))
display(companies.head())
display(companies["Accounts_AccountCategory"].value_counts(dropna=False).rename_axis("Accounts_AccountCategory").reset_index(name="companies"))

Selected companies: 100000
Unique target company numbers: 100000


,CompanyNumber,CompanyName,CompanyStatus,CompanyCategory,CountryOfOrigin,RegAddress_Country,RegAddress_PostTown,RegAddress_PostCode,IncorporationDate,primary_sic_code,...,Accounts_AccountCategory,Accounts_LastMadeUpDate,Accounts_NextDueDate,Mortgages_NumMortOutstanding,Mortgages_NumMortCharges,has_outstanding_charges,company_age_years,CountryOfOrigin_clean,is_uk_company,CompanyNumber_norm
0,13209628,TICKETY BOO TEETH LIMITED,Active,Private Limited Company,United Kingdom,UNITED KINGDOM,BASINGSTOKE,RG24 8PE,2021-02-18,86230,...,UNAUDITED ABRIDGED,2025-06-30,2027-03-31,0,0,False,5.3,united kingdom,True,13209628
1,13887674,TECH INFUSION LIMITED,Active,Private Limited Company,United Kingdom,ENGLAND,HALIFAX,HX1 5LT,2022-02-02,62020,...,TOTAL EXEMPTION FULL,2025-02-28,2026-11-30,0,0,False,4.4,united kingdom,True,13887674
2,13408899,THE SCC ACADEMY LIMITED,Active,Private Limited Company,United Kingdom,ENGLAND,WARWICKSHIRE,CV37 6YX,2021-05-19,62090,...,TOTAL EXEMPTION FULL,2025-04-05,2027-01-05,0,0,False,5.1,united kingdom,True,13408899
3,SC374368,GRACEFRUIT LIMITED,Active,Private Limited Company,United Kingdom,NaN,LONGCROFT,FK4 1QL,2010-03-08,46450,...,TOTAL EXEMPTION FULL,2025-03-31,2026-12-31,0,0,False,16.3,united kingdom,True,SC374368
4,13675979,LINHAM LIMITED,Active,Private Limited Company,United Kingdom,UNITED KINGDOM,HOLYWELL,CH8 7LH,2021-10-13,47110,...,TOTAL EXEMPTION FULL,2025-10-31,2027-07-31,1,1,True,4.7,united kingdom,True,13675979


,Accounts_AccountCategory,companies
0,MICRO ENTITY,53241
1,TOTAL EXEMPTION FULL,36917
2,UNAUDITED ABRIDGED,4515
3,FULL,1834
4,SMALL,1781
5,AUDIT EXEMPTION SUBSIDIARY,902
6,GROUP,627
7,MEDIUM,183


## 4. Build Accounts Match Index

In [4]:
zip_paths = sorted(ACCOUNTS_ZIP_DIR.glob("*.zip"))
print("ZIP files found:", len(zip_paths))
for p in zip_paths:
    print(p.name)

match_rows = []
zip_file_summaries = []
start = time.time()

for zip_idx, zip_path in enumerate(zip_paths, start=1):
    t0 = time.time()
    parsed_entries = 0
    matched_entries = 0
    with zipfile.ZipFile(zip_path) as zf:
        entries = zf.infolist()
        for entry in entries:
            if entry.is_dir():
                continue
            parsed = parse_member_name(entry.filename)
            if not parsed:
                continue
            parsed_entries += 1
            company_number, period_end, file_format = parsed
            if company_number in target_company_numbers:
                matched_entries += 1
                match_rows.append({
                    "CompanyNumber_norm": company_number,
                    "period_end": period_end,
                    "source_zip": zip_path.name,
                    "internal_filename": entry.filename,
                    "file_format": file_format,
                    "file_size": entry.file_size,
                })
    zip_file_summaries.append({
        "source_zip": zip_path.name,
        "parsed_entries": parsed_entries,
        "matched_entries": matched_entries,
        "elapsed_seconds": round(time.time() - t0, 2),
    })
    print(f"[{zip_idx}/{len(zip_paths)}] {zip_path.name}: parsed={parsed_entries:,}, matched={matched_entries:,}, elapsed={time.time() - t0:.1f}s")

match_index = pd.DataFrame(match_rows)
if not match_index.empty:
    match_index = match_index.sort_values(["CompanyNumber_norm", "period_end", "source_zip", "internal_filename"])
    match_index.to_csv(MATCH_INDEX_CSV, index=False, encoding="utf-8-sig")

zip_file_summary = pd.DataFrame(zip_file_summaries)
print("Matched account files:", len(match_index))
print("Matched companies:", match_index["CompanyNumber_norm"].nunique() if not match_index.empty else 0)
print("Index written:", MATCH_INDEX_CSV)
display(zip_file_summary)
display(match_index.head())

ZIP files found: 24
Accounts_Monthly_Data-April2025.zip
Accounts_Monthly_Data-April2026.zip
Accounts_Monthly_Data-August2024.zip
Accounts_Monthly_Data-August2025.zip
Accounts_Monthly_Data-December2024.zip
Accounts_Monthly_Data-December2025.zip
Accounts_Monthly_Data-February2025.zip
Accounts_Monthly_Data-February2026.zip
Accounts_Monthly_Data-January2025.zip
Accounts_Monthly_Data-January2026.zip
Accounts_Monthly_Data-July2024.zip
Accounts_Monthly_Data-July2025.zip
Accounts_Monthly_Data-June2025.zip
Accounts_Monthly_Data-June2026.zip
Accounts_Monthly_Data-March2025.zip
Accounts_Monthly_Data-March2026.zip
Accounts_Monthly_Data-May2025.zip
Accounts_Monthly_Data-May2026.zip
Accounts_Monthly_Data-November2024.zip
Accounts_Monthly_Data-November2025.zip
Accounts_Monthly_Data-October2024.zip
Accounts_Monthly_Data-October2025.zip
Accounts_Monthly_Data-September2024.zip
Accounts_Monthly_Data-September2025.zip
[1/24] Accounts_Monthly_Data-April2025.zip: parsed=264,543, matched=6,021, elapsed=9.7s


,source_zip,parsed_entries,matched_entries,elapsed_seconds
0,Accounts_Monthly_Data-April2025.zip,264543,6021,9.70
1,Accounts_Monthly_Data-April2026.zip,263094,6665,7.41
2,Accounts_Monthly_Data-August2024.zip,258179,5574,7.28
3,Accounts_Monthly_Data-August2025.zip,261885,6259,6.20
4,Accounts_Monthly_Data-December2024.zip,487794,11819,12.79
5,Accounts_Monthly_Data-December2025.zip,499784,13014,12.46
6,Accounts_Monthly_Data-February2025.zip,250276,5694,6.10
7,Accounts_Monthly_Data-February2026.zip,258471,6361,6.21
8,Accounts_Monthly_Data-January2025.zip,287375,6339,6.85
9,Accounts_Monthly_Data-January2026.zip,289256,7089,5.80


,CompanyNumber_norm,period_end,source_zip,internal_filename,file_format,file_size
49352,00031641,2024-05-31,Accounts_Monthly_Data-February2025.zip,Prod224_3600_00031641_20240531.html,html,82779
67746,00031641,2025-05-31,Accounts_Monthly_Data-January2026.zip,Prod224_2601_00031641_20250531.html,html,78037
12686,00038191,2024-03-31,Accounts_Monthly_Data-August2024.zip,Prod224_2466_00038191_20240331.html,html,344626
18260,00038191,2025-03-31,Accounts_Monthly_Data-August2025.zip,Prod224_2508_00038191_20250331.html,html,342343
158685,00041365,2023-12-31,Accounts_Monthly_Data-September2024.zip,Prod224_2476_00041365_20231231.html,html,117245


## 5. Parse Matched Accounts Files

In [5]:
feature_rows = []
fact_rows = []
parse_error_rows = []

if match_index.empty:
    raise ValueError("No matched accounts files found. Check company number normalisation and ZIP directory.")

parse_df = match_index.copy()
if MAX_PARSE_FILES is not None:
    parse_df = parse_df.head(MAX_PARSE_FILES).copy()

rows_by_zip = {name: g for name, g in parse_df.groupby("source_zip", sort=False)}
parsed_count = 0
start = time.time()

for zip_idx, (zip_name, rows) in enumerate(rows_by_zip.items(), start=1):
    zip_path = ACCOUNTS_ZIP_DIR / zip_name
    t0 = time.time()
    with zipfile.ZipFile(zip_path) as zf:
        for _, r in rows.iterrows():
            parsed_count += 1
            if parsed_count % 1000 == 0:
                print(f"Parsed {parsed_count:,}/{len(parse_df):,} matched files...")
            try:
                text = safe_read_text_from_zip(zf, r["internal_filename"])
                facts = parse_financial_facts(
                    text=text,
                    company_number=r["CompanyNumber_norm"],
                    period_end=r["period_end"],
                    source_zip=r["source_zip"],
                    internal_filename=r["internal_filename"],
                )
                selected = select_metric_values(facts)
                row = {
                    "CompanyNumber_norm": r["CompanyNumber_norm"],
                    "period_end": r["period_end"],
                    "source_zip": r["source_zip"],
                    "internal_filename": r["internal_filename"],
                    "file_format": r["file_format"],
                    "file_size": r["file_size"],
                    "parsed_ok": True,
                    "facts_extracted_count": len(facts),
                }
                row.update(selected)
                row["available_core_proxy_field_count"] = sum(pd.notna(row.get(c)) for c in CORE_PROXY_FIELDS)
                row["available_any_financial_field_count"] = sum(pd.notna(row.get(c)) for c in set(CORE_PROXY_FIELDS + ["turnover", "fixed_assets", "profit_loss", "creditors_within_one_year", "creditors_after_one_year"]))
                row["financial_evidence_tier"] = assign_evidence_tier(row)
                feature_rows.append(row)
                if WRITE_FACTS_LONG and facts:
                    fact_rows.extend(facts)
            except Exception as exc:
                parse_error_rows.append({
                    "CompanyNumber_norm": r["CompanyNumber_norm"],
                    "period_end": r["period_end"],
                    "source_zip": r["source_zip"],
                    "internal_filename": r["internal_filename"],
                    "error": repr(exc),
                })
    print(f"[{zip_idx}/{len(rows_by_zip)}] parsed {zip_name}: files={len(rows):,}, elapsed={time.time() - t0:.1f}s")

filing_features = pd.DataFrame(feature_rows)
parse_errors = pd.DataFrame(parse_error_rows)

if not filing_features.empty:
    filing_features = filing_features.sort_values(["CompanyNumber_norm", "period_end", "source_zip", "internal_filename"])
    filing_features.to_csv(FILING_FEATURES_CSV, index=False, encoding="utf-8-sig")
if WRITE_FACTS_LONG and fact_rows:
    facts_long = pd.DataFrame(fact_rows)
    facts_long.to_csv(FACTS_LONG_CSV, index=False, encoding="utf-8-sig")
if not parse_errors.empty:
    parse_errors.to_csv(OUTPUT_DIR / "accounts_parse_errors_100k.csv", index=False, encoding="utf-8-sig")

print("Parsed filing rows:", len(filing_features))
print("Parse errors:", len(parse_errors))
print("Filing features written:", FILING_FEATURES_CSV)
display(filing_features.head())

Parsed 1,000/176,413 matched files...
Parsed 2,000/176,413 matched files...
Parsed 3,000/176,413 matched files...
Parsed 4,000/176,413 matched files...
Parsed 5,000/176,413 matched files...
[1/24] parsed Accounts_Monthly_Data-February2025.zip: files=5,694, elapsed=139.9s
Parsed 6,000/176,413 matched files...
Parsed 7,000/176,413 matched files...
Parsed 8,000/176,413 matched files...
Parsed 9,000/176,413 matched files...
Parsed 10,000/176,413 matched files...
Parsed 11,000/176,413 matched files...
Parsed 12,000/176,413 matched files...
[2/24] parsed Accounts_Monthly_Data-January2026.zip: files=7,089, elapsed=177.2s
Parsed 13,000/176,413 matched files...
Parsed 14,000/176,413 matched files...
Parsed 15,000/176,413 matched files...
Parsed 16,000/176,413 matched files...
Parsed 17,000/176,413 matched files...
Parsed 18,000/176,413 matched files...
[3/24] parsed Accounts_Monthly_Data-August2024.zip: files=5,574, elapsed=160.4s
Parsed 19,000/176,413 matched files...
Parsed 20,000/176,413 mat

,CompanyNumber_norm,period_end,source_zip,internal_filename,file_format,file_size,parsed_ok,facts_extracted_count,cash,creditors_total,...,equity,fixed_assets,net_assets_liabilities,net_current_assets_liabilities,profit_loss,total_assets_less_current_liabilities,available_core_proxy_field_count,available_any_financial_field_count,financial_evidence_tier,turnover
0,00031641,2024-05-31,Accounts_Monthly_Data-February2025.zip,Prod224_3600_00031641_20240531.html,html,82779,True,42,62524.0,51454.0,...,1078840.0,1129269.0,1078840.0,11271.0,70136.0,1140540.0,7,9,T2_balance_sheet_rich,NaN
5694,00031641,2025-05-31,Accounts_Monthly_Data-January2026.zip,Prod224_2601_00031641_20250531.html,html,78037,True,32,47563.0,56157.0,...,1154787.0,1232281.0,1154787.0,-8594.0,NaN,1223687.0,7,8,T2_balance_sheet_rich,NaN
12783,00038191,2024-03-31,Accounts_Monthly_Data-August2024.zip,Prod224_2466_00038191_20240331.html,html,344626,True,38,461714.0,6325.0,...,2116149.0,898233.0,2116149.0,1283786.0,NaN,2182019.0,7,8,T2_balance_sheet_rich,NaN
18357,00038191,2025-03-31,Accounts_Monthly_Data-August2025.zip,Prod224_2508_00038191_20250331.html,html,342343,True,38,547259.0,6670.0,...,2264835.0,894468.0,2264835.0,1435343.0,NaN,2329811.0,7,8,T2_balance_sheet_rich,NaN
24616,00041365,2023-12-31,Accounts_Monthly_Data-September2024.zip,Prod224_2476_00041365_20231231.html,html,117245,True,44,3244.0,50462.0,...,490008.0,NaN,490008.0,-1673134.0,-149203.0,893428.0,7,8,T2_balance_sheet_rich,NaN


## 6. Company-Level Summary and Evidence Tier Distribution

In [6]:
if filing_features.empty:
    raise ValueError("No filing features were parsed.")

evidence_rank = {
    "T1_observed_turnover": 1,
    "T2_balance_sheet_rich": 2,
    "T3_balance_sheet_partial": 3,
    "T4_account_category_only": 4,
}

ff = filing_features.copy()
ff["evidence_rank"] = ff["financial_evidence_tier"].map(evidence_rank).fillna(99)
ff["has_turnover"] = ff["turnover"].notna()
ff["has_useful_financial_evidence"] = ff["financial_evidence_tier"].isin(["T1_observed_turnover", "T2_balance_sheet_rich", "T3_balance_sheet_partial"])
ff["period_end_dt"] = pd.to_datetime(ff["period_end"], errors="coerce")

# Latest account period, and if duplicate filings exist for the same period, keep the strongest evidence.
latest_idx = (
    ff.sort_values(["CompanyNumber_norm", "period_end_dt", "evidence_rank"], ascending=[True, True, False])
    .groupby("CompanyNumber_norm")
    .tail(1)
    .index
)
latest_filing = ff.loc[latest_idx].copy()

# Best evidence filing, and if multiple filings have the same evidence tier, keep the latest one.
best_idx = (
    ff.sort_values(["CompanyNumber_norm", "evidence_rank", "period_end_dt"], ascending=[True, True, False])
    .groupby("CompanyNumber_norm")
    .head(1)
    .index
)
best_filing = ff.loc[best_idx].copy()

company_agg = ff.groupby("CompanyNumber_norm").agg(
    matched_account_files=("internal_filename", "count"),
    distinct_account_periods=("period_end", "nunique"),
    parsed_financial_periods=("period_end", lambda s: s[ff.loc[s.index, "has_useful_financial_evidence"]].nunique()),
    turnover_periods=("period_end", lambda s: s[ff.loc[s.index, "has_turnover"]].nunique()),
    first_period_end=("period_end_dt", "min"),
    latest_period_end=("period_end_dt", "max"),
    has_any_turnover=("has_turnover", "max"),
    has_any_useful_financial_evidence=("has_useful_financial_evidence", "max"),
).reset_index()

latest_cols = ["CompanyNumber_norm", "period_end", "financial_evidence_tier", "turnover", "available_core_proxy_field_count", "available_any_financial_field_count"]
latest_small = latest_filing[latest_cols].rename(columns={
    "period_end": "latest_period_with_account",
    "financial_evidence_tier": "latest_financial_evidence_tier",
    "turnover": "latest_turnover",
    "available_core_proxy_field_count": "latest_core_proxy_field_count",
    "available_any_financial_field_count": "latest_any_financial_field_count",
})
best_small = best_filing[latest_cols].rename(columns={
    "period_end": "best_evidence_period_end",
    "financial_evidence_tier": "best_financial_evidence_tier",
    "turnover": "best_observed_turnover",
    "available_core_proxy_field_count": "best_core_proxy_field_count",
    "available_any_financial_field_count": "best_any_financial_field_count",
})

company_summary = (
    companies
    .merge(company_agg, on="CompanyNumber_norm", how="left")
    .merge(latest_small, on="CompanyNumber_norm", how="left")
    .merge(best_small, on="CompanyNumber_norm", how="left")
)

company_summary["has_matched_accounts"] = company_summary["matched_account_files"].fillna(0).astype(float) > 0
company_summary["has_two_plus_matched_account_periods"] = company_summary["distinct_account_periods"].fillna(0).astype(float) >= 2
company_summary["has_two_plus_financial_evidence_periods"] = company_summary["parsed_financial_periods"].fillna(0).astype(float) >= 2
company_summary["has_two_plus_turnover_periods"] = company_summary["turnover_periods"].fillna(0).astype(float) >= 2

company_summary.to_csv(COMPANY_SUMMARY_CSV, index=False, encoding="utf-8-sig")

tier_filing_dist = ff["financial_evidence_tier"].value_counts(dropna=False).rename_axis("financial_evidence_tier").reset_index(name="filing_count")
tier_company_best_dist = company_summary["best_financial_evidence_tier"].fillna("No_matched_accounts").value_counts(dropna=False).rename_axis("best_financial_evidence_tier").reset_index(name="company_count")
tier_company_latest_dist = company_summary["latest_financial_evidence_tier"].fillna("No_matched_accounts").value_counts(dropna=False).rename_axis("latest_financial_evidence_tier").reset_index(name="company_count")

tier_distribution = pd.concat([
    tier_filing_dist.assign(level="filing_level").rename(columns={"financial_evidence_tier": "tier", "filing_count": "count"}),
    tier_company_best_dist.assign(level="company_best_evidence").rename(columns={"best_financial_evidence_tier": "tier", "company_count": "count"}),
    tier_company_latest_dist.assign(level="company_latest_account").rename(columns={"latest_financial_evidence_tier": "tier", "company_count": "count"}),
], ignore_index=True)
tier_distribution.to_csv(TIER_DISTRIBUTION_CSV, index=False, encoding="utf-8-sig")

turnover_companies = company_summary[company_summary["has_any_turnover"].fillna(False)].copy()
turnover_companies.to_csv(TURNOVER_COMPANIES_CSV, index=False, encoding="utf-8-sig")

multi_period_companies = company_summary[
    company_summary["has_two_plus_matched_account_periods"] |
    company_summary["has_two_plus_financial_evidence_periods"] |
    company_summary["has_two_plus_turnover_periods"]
].copy()
multi_period_companies.to_csv(MULTI_PERIOD_COMPANIES_CSV, index=False, encoding="utf-8-sig")

print("Company summary written:", COMPANY_SUMMARY_CSV)
print("Turnover companies:", len(turnover_companies))
print("2+ matched account periods:", int(company_summary["has_two_plus_matched_account_periods"].sum()))
print("2+ financial evidence periods:", int(company_summary["has_two_plus_financial_evidence_periods"].sum()))
print("2+ turnover periods:", int(company_summary["has_two_plus_turnover_periods"].sum()))
display(tier_distribution)
display(company_summary.head())

C:\Users\RoHenry\AppData\Local\Temp\ipykernel_1660\42370815.py:87: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  turnover_companies = company_summary[company_summary["has_any_turnover"].fillna(False)].copy()


Company summary written: E:\000硕士毕设\公司+财务数据匹配\financial_features_company_summary_100k.csv
Turnover companies: 3183
2+ matched account periods: 76912
2+ financial evidence periods: 76423
2+ turnover periods: 1198


,tier,count,level
0,T2_balance_sheet_rich,163937,filing_level
1,T3_balance_sheet_partial,7069,filing_level
2,T1_observed_turnover,4431,filing_level
3,T4_account_category_only,976,filing_level
4,T2_balance_sheet_rich,87837,company_best_evidence
5,No_matched_accounts,5688,company_best_evidence
6,T1_observed_turnover,3183,company_best_evidence
7,T3_balance_sheet_partial,2850,company_best_evidence
8,T4_account_category_only,442,company_best_evidence
9,T2_balance_sheet_rich,88095,company_latest_account


,CompanyNumber,CompanyName,CompanyStatus,CompanyCategory,CountryOfOrigin,RegAddress_Country,RegAddress_PostTown,RegAddress_PostCode,IncorporationDate,primary_sic_code,...,latest_any_financial_field_count,best_evidence_period_end,best_financial_evidence_tier,best_observed_turnover,best_core_proxy_field_count,best_any_financial_field_count,has_matched_accounts,has_two_plus_matched_account_periods,has_two_plus_financial_evidence_periods,has_two_plus_turnover_periods
0,13209628,TICKETY BOO TEETH LIMITED,Active,Private Limited Company,United Kingdom,UNITED KINGDOM,BASINGSTOKE,RG24 8PE,2021-02-18,86230,...,8.0,2025-06-30,T2_balance_sheet_rich,NaN,7.0,8.0,True,True,True,False
1,13887674,TECH INFUSION LIMITED,Active,Private Limited Company,United Kingdom,ENGLAND,HALIFAX,HX1 5LT,2022-02-02,62020,...,4.0,2025-02-28,T2_balance_sheet_rich,NaN,4.0,4.0,True,True,True,False
2,13408899,THE SCC ACADEMY LIMITED,Active,Private Limited Company,United Kingdom,ENGLAND,WARWICKSHIRE,CV37 6YX,2021-05-19,62090,...,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,False
3,SC374368,GRACEFRUIT LIMITED,Active,Private Limited Company,United Kingdom,NaN,LONGCROFT,FK4 1QL,2010-03-08,46450,...,8.0,2025-03-31,T2_balance_sheet_rich,NaN,7.0,8.0,True,False,False,False
4,13675979,LINHAM LIMITED,Active,Private Limited Company,United Kingdom,UNITED KINGDOM,HOLYWELL,CH8 7LH,2021-10-13,47110,...,6.0,2025-10-31,T2_balance_sheet_rich,NaN,5.0,6.0,True,True,True,False


## 7. Coverage by Account Category and Sector

In [7]:
def coverage_table(group_col):
    g = company_summary.groupby(group_col, dropna=False)
    out = g.agg(
        companies=("CompanyNumber_norm", "count"),
        matched_companies=("has_matched_accounts", "sum"),
        turnover_companies=("has_any_turnover", "sum"),
        useful_financial_evidence_companies=("has_any_useful_financial_evidence", "sum"),
        two_plus_matched_period_companies=("has_two_plus_matched_account_periods", "sum"),
        two_plus_financial_evidence_period_companies=("has_two_plus_financial_evidence_periods", "sum"),
        two_plus_turnover_period_companies=("has_two_plus_turnover_periods", "sum"),
    ).reset_index()
    for col in [
        "matched_companies",
        "turnover_companies",
        "useful_financial_evidence_companies",
        "two_plus_matched_period_companies",
        "two_plus_financial_evidence_period_companies",
        "two_plus_turnover_period_companies",
    ]:
        out[col + "_rate"] = out[col] / out["companies"]
    return out.sort_values(["companies"], ascending=False)

category_coverage = coverage_table("Accounts_AccountCategory")
sector_coverage = coverage_table("primary_sector")

category_coverage.to_csv(CATEGORY_COVERAGE_CSV, index=False, encoding="utf-8-sig")
sector_coverage.to_csv(SECTOR_COVERAGE_CSV, index=False, encoding="utf-8-sig")

display(category_coverage)
display(sector_coverage)

,Accounts_AccountCategory,companies,matched_companies,turnover_companies,useful_financial_evidence_companies,two_plus_matched_period_companies,two_plus_financial_evidence_period_companies,two_plus_turnover_period_companies,matched_companies_rate,turnover_companies_rate,useful_financial_evidence_companies_rate,two_plus_matched_period_companies_rate,two_plus_financial_evidence_period_companies_rate,two_plus_turnover_period_companies_rate
4,MICRO ENTITY,53241,52667,1872,52457,42473,42218,643,0.989219,0.035161,0.985275,0.797750,0.792960,0.012077
6,TOTAL EXEMPTION FULL,36917,35186,826,34990,29289,29078,281,0.953111,0.022375,0.947802,0.793374,0.787659,0.007612
7,UNAUDITED ABRIDGED,4515,4433,34,4431,3693,3690,0,0.981838,0.00753,0.981395,0.817940,0.817276,0.000000
1,FULL,1834,570,235,541,405,387,142,0.310796,0.128135,0.294984,0.220829,0.211014,0.077426
5,SMALL,1781,997,52,994,750,748,15,0.559798,0.029197,0.558113,0.421112,0.419989,0.008422
0,AUDIT EXEMPTION SUBSIDIARY,902,59,3,59,1,1,0,0.065410,0.003326,0.06541,0.001109,0.001109,0.000000
2,GROUP,627,246,23,244,172,172,13,0.392344,0.036683,0.389155,0.274322,0.274322,0.020734
3,MEDIUM,183,154,138,154,129,129,104,0.841530,0.754098,0.84153,0.704918,0.704918,0.568306


,primary_sector,companies,matched_companies,turnover_companies,useful_financial_evidence_companies,two_plus_matched_period_companies,two_plus_financial_evidence_period_companies,two_plus_turnover_period_companies,matched_companies_rate,turnover_companies_rate,useful_financial_evidence_companies_rate,two_plus_matched_period_companies_rate,two_plus_financial_evidence_period_companies_rate,two_plus_turnover_period_companies_rate
6,"Technology, legal & professional",26432,24988,845,24889,20592,20472,295,0.945369,0.031969,0.941624,0.779056,0.774516,0.011161
5,Real Estate,24727,23606,406,23416,19467,19265,141,0.954665,0.016419,0.946981,0.787277,0.779108,0.005702
7,Wholesale & Retail,17841,17193,629,17176,13641,13613,253,0.963679,0.035256,0.962726,0.764587,0.763018,0.014181
1,Fast growth & emerging sector,9761,9334,423,9299,7466,7409,165,0.956254,0.043336,0.952669,0.764881,0.759041,0.016904
2,Healthcare,7913,7329,262,7291,6014,5980,91,0.926197,0.03311,0.921395,0.760015,0.755718,0.011500
3,Manufacturing,7066,6531,302,6520,5478,5466,129,0.924285,0.04274,0.922729,0.775262,0.773564,0.018256
4,"Public sector, education & charities",4697,3841,267,3789,2989,2957,100,0.817756,0.056845,0.806685,0.636364,0.629551,0.021290
0,Agriculture,1563,1490,49,1490,1265,1261,24,0.953295,0.03135,0.953295,0.809341,0.806782,0.015355


## 8. Summary JSON

In [8]:
summary = {
    "input_company_csv": str(COMPANY_CSV),
    "accounts_zip_dir": str(ACCOUNTS_ZIP_DIR),
    "output_dir": str(OUTPUT_DIR),
    "selected_companies": int(len(companies)),
    "zip_files": int(len(zip_paths)),
    "matched_account_files": int(len(match_index)),
    "matched_companies": int(company_summary["has_matched_accounts"].sum()),
    "matched_company_rate": float(company_summary["has_matched_accounts"].mean()),
    "turnover_companies": int(company_summary["has_any_turnover"].fillna(False).sum()),
    "turnover_company_rate_all_selected": float(company_summary["has_any_turnover"].fillna(False).mean()),
    "turnover_company_rate_matched": float(
        company_summary.loc[company_summary["has_matched_accounts"], "has_any_turnover"].fillna(False).mean()
    ) if company_summary["has_matched_accounts"].any() else None,
    "companies_with_two_plus_matched_account_periods": int(company_summary["has_two_plus_matched_account_periods"].sum()),
    "companies_with_two_plus_financial_evidence_periods": int(company_summary["has_two_plus_financial_evidence_periods"].sum()),
    "companies_with_two_plus_turnover_periods": int(company_summary["has_two_plus_turnover_periods"].sum()),
    "parse_errors": int(len(parse_errors)),
    "tier_distribution_file": str(TIER_DISTRIBUTION_CSV),
    "company_summary_file": str(COMPANY_SUMMARY_CSV),
    "turnover_companies_file": str(TURNOVER_COMPANIES_CSV),
    "multi_period_companies_file": str(MULTI_PERIOD_COMPANIES_CSV),
}

with open(SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(json.dumps(summary, ensure_ascii=False, indent=2))

{
  "input_company_csv": "E:\\000硕士毕设\\Haoran_CHENG\\01 Data PreProcessing\\01_CompaniesSelected\\UKcompanies_active_account_category_sample_100k.csv",
  "accounts_zip_dir": "E:\\000硕士毕设\\财务数据\\Local Large Data\\Accounts Data_2024.7_2026.6",
  "output_dir": "E:\\000硕士毕设\\公司+财务数据匹配",
  "selected_companies": 100000,
  "zip_files": 24,
  "matched_account_files": 176413,
  "matched_companies": 94312,
  "matched_company_rate": 0.94312,
  "turnover_companies": 3183,
  "turnover_company_rate_all_selected": 0.03183,
  "turnover_company_rate_matched": 0.03374968190686233,
  "companies_with_two_plus_matched_account_periods": 76912,
  "companies_with_two_plus_financial_evidence_periods": 76423,
  "companies_with_two_plus_turnover_periods": 1198,
  "parse_errors": 0,
  "tier_distribution_file": "E:\\000硕士毕设\\公司+财务数据匹配\\financial_evidence_tier_distribution_100k.csv",
  "company_summary_file": "E:\\000硕士毕设\\公司+财务数据匹配\\financial_features_company_summary_100k.csv",
  "turnover_companies_file": "E:\\00

C:\Users\RoHenry\AppData\Local\Temp\ipykernel_1660\3806283227.py:10: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  "turnover_companies": int(company_summary["has_any_turnover"].fillna(False).sum()),
C:\Users\RoHenry\AppData\Local\Temp\ipykernel_1660\3806283227.py:11: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  "turnover_company_rate_all_selected": float(company_summary["has_any_turnover"].fillna(False).mean()),
C:\Users\RoHenry\AppData\Local\Temp\ipykernel_1660\3806283227.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill 